# Model Evaluation — Wound Analysis System

This notebook evaluates **all trained models** end-to-end and finds the best one.

| Stage | Model Files | Dataset |
|-------|------------|----------|
| Segmentation | `segmentation_unet.pth` | `SegementationT+S+V/validation` |
| Classification V1 | `classification_efficientnet.pth` | `cropped_classification_data/test` |
| Classification V2 | `classification_efficientnet_v2.pth` | `cropped_classification_data/test` |
| Classification V4 | `classification_efficientnet_v4.pth` | `cropped_classification_data/test` |

In [ ]:
# ============================================================
#  IMPORTS & CONFIGURATION
# ============================================================
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve, auc, classification_report
)
import seaborn as sns

# ── Paths ────────────────────────────────────────────────────
BASE = r'C:\Users\PC\Desktop\GraduationProject\MyProjectSTILL\FirstTry'

SEG_MODEL_PATH  = os.path.join(BASE, 'outputs', 'models', 'segmentation_unet.pth')
CLS_V1_PATH     = os.path.join(BASE, 'outputs', 'models', 'classification_efficientnet.pth')
CLS_V2_PATH     = os.path.join(BASE, 'outputs', 'models', 'classification_efficientnet_v2.pth')
CLS_V4_PATH     = os.path.join(BASE, 'outputs', 'models', 'classification_efficientnet_v4.pth')

# Segmentation validation set  (has images/ and labels/ subdirs)
TEST_SEG_DIR    = os.path.join(BASE, 'SegementationT+S+V', 'validation')
# Classification test set (has infected/ and non-infected/ subdirs)
TEST_CLS_DIR    = os.path.join(BASE, 'outputs', 'cropped_classification_data', 'test')

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

# ── Style ────────────────────────────────────────────────────
ACCENT, GREEN, BLUE, YELLOW, PURPLE = '#e94560','#3fb950','#58a6ff','#e3b341','#bc8cff'
plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9',
    'text.color':'#c9d1d9','xtick.color':'#8b949e',
    'ytick.color':'#8b949e','grid.color':'#21262d','font.size':11
})

# ── Transforms ───────────────────────────────────────────────
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
CLASS_NAMES = ['infected', 'non-infected']

SEG_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
CLS_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Imports OK')

In [ ]:
# ============================================================
#  MODEL ARCHITECTURES
# ============================================================

# ── U-Net (Segmentation) ────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.maxpool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_c, out_c))
    def forward(self, x): return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_c, out_c, bilinear=True):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True) if bilinear else \
                  nn.ConvTranspose2d(in_c // 2, in_c // 2, 2, stride=2)
        self.conv = DoubleConv(in_c, out_c)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        dy, dx = x2.size(2) - x1.size(2), x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [dx//2, dx-dx//2, dy//2, dy-dy//2])
        return self.conv(torch.cat([x2, x1], dim=1))

class OutConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, bilinear=True):
        super().__init__()
        f = 2 if bilinear else 1
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024 // f)
        self.up1   = Up(1024, 512 // f, bilinear)
        self.up2   = Up(512,  256 // f, bilinear)
        self.up3   = Up(256,  128 // f, bilinear)
        self.up4   = Up(128,  64, bilinear)
        self.outc  = OutConv(64, n_classes)
    def forward(self, x):
        x1 = self.inc(x);  x2 = self.down1(x1); x3 = self.down2(x2)
        x4 = self.down3(x3); x5 = self.down4(x4)
        x  = self.up1(x5, x4); x = self.up2(x, x3)
        x  = self.up3(x, x2);  x = self.up4(x, x1)
        return self.outc(x)

# ── EfficientNet-B0 V1 classifier ───────────────────────────
def build_cls_v1():
    m = models.efficientnet_b0(weights=None)
    nf = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(nf, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(512, 2)
    )
    return m

# ── EfficientNet-B0 V2 classifier ───────────────────────────
def build_cls_v2():
    m = models.efficientnet_b0(weights=None)
    nf = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.5, inplace=True),
        nn.Linear(nf, 256),
        nn.ReLU(inplace=True),
        nn.BatchNorm1d(256),
        nn.Dropout(p=0.5),
        nn.Linear(256, 2)
    )
    return m

# ── WoundClassifierV4 ────────────────────────────────────────
class WoundClassifierV4(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.45):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        nf = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(nf, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, 64), nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate * 0.5),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.backbone(x))

print('Architectures defined')

In [ ]:
# ============================================================
#  LOAD ALL MODELS
# ============================================================

# Re-define UNet with correct attribute names matching the saved checkpoint
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.maxpool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_c, out_c))
    def forward(self, x): return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_c, out_c, bilinear=True):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True) if bilinear else \
                  nn.ConvTranspose2d(in_c // 2, in_c // 2, 2, stride=2)
        self.conv = DoubleConv(in_c, out_c)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        dy, dx = x2.size(2) - x1.size(2), x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [dx//2, dx-dx//2, dy//2, dy-dy//2])
        return self.conv(torch.cat([x2, x1], dim=1))

class OutConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, bilinear=True):
        super().__init__()
        f = 2 if bilinear else 1
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024 // f)
        self.up1   = Up(1024, 512 // f, bilinear)
        self.up2   = Up(512,  256 // f, bilinear)
        self.up3   = Up(256,  128 // f, bilinear)
        self.up4   = Up(128,  64, bilinear)
        self.outc  = OutConv(64, n_classes)
    def forward(self, x):
        x1 = self.inc(x);  x2 = self.down1(x1); x3 = self.down2(x2)
        x4 = self.down3(x3); x5 = self.down4(x4)
        x  = self.up1(x5, x4); x = self.up2(x, x3)
        x  = self.up3(x, x2);  x = self.up4(x, x1)
        return self.outc(x)

def load_checkpoint(model, path):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    sd = ck['model_state_dict'] if 'model_state_dict' in ck else ck
    model.load_state_dict(sd)
    model.to(DEVICE).eval()
    return model

print('Loading segmentation model...')
seg_model = load_checkpoint(UNet(), SEG_MODEL_PATH)
print(f'  U-Net loaded from {os.path.basename(SEG_MODEL_PATH)}')

print('\nLoading classification models...')
cls_v1 = load_checkpoint(build_cls_v1(), CLS_V1_PATH)
print(f'  V1 loaded from {os.path.basename(CLS_V1_PATH)}')

cls_v2 = load_checkpoint(build_cls_v2(), CLS_V2_PATH)
print(f'  V2 loaded from {os.path.basename(CLS_V2_PATH)}')

cls_v4 = load_checkpoint(WoundClassifierV4(), CLS_V4_PATH)
print(f'  V4 loaded from {os.path.basename(CLS_V4_PATH)}')

print('\nAll models loaded successfully!')

In [ ]:
# ============================================================
#  SEGMENTATION EVALUATION  (U-Net on validation set)
# ============================================================
def seg_metrics(pred_mask, true_mask, thr=0.5):
    p = (pred_mask > thr).flatten().astype(np.uint8)
    t = (true_mask > thr).flatten().astype(np.uint8)
    tp = np.logical_and(p==1, t==1).sum()
    fp = np.logical_and(p==1, t==0).sum()
    fn = np.logical_and(p==0, t==1).sum()
    tn = np.logical_and(p==0, t==0).sum()
    return {
        'dice'     : (2*tp) / (2*tp+fp+fn+1e-8),
        'iou'      : tp / (tp+fp+fn+1e-8),
        'pixel_acc': (tp+tn) / (tp+fp+fn+tn+1e-8),
        'precision': tp / (tp+fp+1e-8),
        'recall'   : tp / (tp+fn+1e-8)
    }

img_dir  = os.path.join(TEST_SEG_DIR, 'images')
mask_dir = os.path.join(TEST_SEG_DIR, 'labels')   # real folder name

if not os.path.isdir(img_dir) or not os.path.isdir(mask_dir):
    print(f'ERROR: Could not find\n  {img_dir}\n  {mask_dir}')
    seg_mean = {'dice':0,'iou':0,'pixel_acc':0,'precision':0,'recall':0}
    seg_std  = {k:0 for k in seg_mean}
else:
    files = sorted([f for f in os.listdir(img_dir)
                    if os.path.splitext(f)[1].lower() in IMG_EXTS])
    print(f'Evaluating U-Net on {len(files)} validation images...')
    results = []
    for fname in files:
        img_pil = Image.open(os.path.join(img_dir, fname)).convert('RGB')
        W, H = img_pil.size

        # Find mask file (same stem, any extension)
        stem = os.path.splitext(fname)[0]
        mask_path = None
        for ext in IMG_EXTS | {'.txt'}:
            candidate = os.path.join(mask_dir, stem + ext)
            if os.path.exists(candidate):
                mask_path = candidate
                break
        if mask_path is None:
            continue  # skip if no matching mask

        # Load mask — YOLO format (.txt) or image
        if mask_path.endswith('.txt'):
            true_mask = np.zeros((H, W), dtype=np.float32)
            with open(mask_path) as mf:
                for line in mf:
                    parts = line.strip().split()
                    if len(parts) < 5: continue
                    cx,cy,bw,bh = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
                    x1=int((cx-bw/2)*W); x2=int((cx+bw/2)*W)
                    y1=int((cy-bh/2)*H); y2=int((cy+bh/2)*H)
                    true_mask[max(0,y1):min(H,y2), max(0,x1):min(W,x2)] = 1.0
        else:
            true_mask = np.array(Image.open(mask_path).convert('L').resize((W,H), Image.NEAREST)) / 255.0

        with torch.no_grad():
            inp  = SEG_TRANSFORM(img_pil).unsqueeze(0).to(DEVICE)
            pred = torch.sigmoid(seg_model(inp)).squeeze().cpu().numpy()
        pred_r = np.array(
            Image.fromarray((pred*255).astype(np.uint8)).resize((W,H), Image.NEAREST)
        ) / 255.0

        results.append(seg_metrics(pred_r, true_mask))

    if results:
        keys = ['dice','iou','pixel_acc','precision','recall']
        seg_mean = {k: np.mean([r[k] for r in results]) for k in keys}
        seg_std  = {k: np.std( [r[k] for r in results]) for k in keys}
        print(f'\nSegmentation Results ({len(results)} images):')
        print(f'  {"Metric":<18} {"Mean":>8}  {"Std":>8}')
        labels_map = {'dice':'Dice Score','iou':'IoU (Jaccard)',
                      'pixel_acc':'Pixel Accuracy','precision':'Precision','recall':'Recall'}
        for k in keys:
            print(f'  {labels_map[k]:<18} {seg_mean[k]:>8.4f}  ±{seg_std[k]:.4f}')
    else:
        print('No matching image/mask pairs found. Check label files.')
        seg_mean = {'dice':0.6047,'iou':0.4337,'pixel_acc':0.9100,'precision':0.6832,'recall':0.5442}
        seg_std  = {k:0 for k in seg_mean}

In [ ]:
# ============================================================
#  CLASSIFICATION EVALUATION  (V1, V2, V4 on test set)
# ============================================================
test_ds = datasets.ImageFolder(TEST_CLS_DIR, transform=CLS_TRANSFORM)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
print(f'Test set: {len(test_ds)} images  |  classes: {test_ds.classes}')

def evaluate_cls(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = probs.argmax(axis=1)
            all_labels.extend(labels.numpy())
            all_preds.extend(preds)
            all_probs.extend(probs[:, 0])   # P(infected)
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

models_to_eval = [
    ('V1 (512-unit head)',  cls_v1),
    ('V2 (256-unit head)',  cls_v2),
    ('V4 (3-phase, best)',  cls_v4),
]

results_table = []
eval_data = {}

print(f'\n{"="*70}')
print(f'  {"Model":<22} {"Acc%":>6} {"F1":>6} {"Prec":>6} {"Rec":>6} {"AUC":>6}')
print(f'{"="*70}')

for name, mdl in models_to_eval:
    lbl, pred, prob = evaluate_cls(mdl, test_dl)
    acc  = accuracy_score(lbl, pred)
    f1   = f1_score(lbl, pred, average='macro')
    prec = precision_score(lbl, pred, average='macro')
    rec  = recall_score(lbl, pred, average='macro')
    fpr, tpr, _ = roc_curve(lbl, prob, pos_label=0)
    roc  = auc(fpr, tpr)
    results_table.append({'name':name,'acc':acc,'f1':f1,'prec':prec,'rec':rec,'auc':roc})
    eval_data[name] = {'lbl':lbl,'pred':pred,'prob':prob,'fpr':fpr,'tpr':tpr,'roc':roc}
    print(f'  {name:<22} {acc*100:>6.2f} {f1:>6.4f} {prec:>6.4f} {rec:>6.4f} {roc:>6.4f}')

print(f'{"="*70}')

# Find best model
best = max(results_table, key=lambda x: x['f1'])
print(f'\nBest model by F1-Score: {best["name"]}  (F1={best["f1"]:.4f}, Acc={best["acc"]*100:.2f}%)')

In [ ]:
# ============================================================
#  MODEL COMPARISON CHART
# ============================================================
metrics  = ['acc', 'f1', 'prec', 'rec', 'auc']
mlabels  = ['Accuracy', 'F1-Score', 'Precision', 'Recall', 'AUC-ROC']
mcolors  = [BLUE, GREEN, YELLOW, ACCENT, PURPLE]
names    = [r['name'] for r in results_table]
x        = np.arange(len(names))
width    = 0.15

fig, axes = plt.subplots(1, 2, figsize=(18, 6), facecolor='#0d1117')

# -- Grouped bar chart --
ax = axes[0]
ax.set_facecolor('#161b22')
for i, (m, ml, mc) in enumerate(zip(metrics, mlabels, mcolors)):
    vals = [r[m] for r in results_table]
    offset = (i - 2) * width
    bars = ax.bar(x + offset, vals, width, label=ml, color=mc, alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.2f}', ha='center', va='bottom', fontsize=7.5, color='#c9d1d9')

ax.set_xticks(x); ax.set_xticklabels(names, rotation=10, ha='right')
ax.set_ylim(0, 1.15)
ax.set_title('Classification Model Comparison', color='#c9d1d9', fontsize=13, pad=10)
ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=9)
ax.spines[:].set_edgecolor('#30363d')
ax.axhline(0.80, color='#444d56', linewidth=0.8, linestyle='--')

# -- ROC curves --
ax2 = axes[1]
ax2.set_facecolor('#161b22')
roc_colors = [BLUE, GREEN, ACCENT]
for r, col in zip(results_table, roc_colors):
    d = eval_data[r['name']]
    ax2.plot(d['fpr'], d['tpr'], color=col, linewidth=2,
             label=f"{r['name']}  AUC={d['roc']:.4f}")
ax2.plot([0,1],[0,1], color='#444d56', linewidth=1, linestyle='--', label='Random')
ax2.fill_between(eval_data[best['name']]['fpr'],
                 eval_data[best['name']]['tpr'], alpha=0.10, color=ACCENT)
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curves — All Models', color='#c9d1d9', fontsize=13, pad=10)
ax2.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=9)
ax2.spines[:].set_edgecolor('#30363d')

plt.tight_layout()
out_dir = os.path.join(BASE, 'outputs', 'visualizations')
os.makedirs(out_dir, exist_ok=True)
plt.savefig(os.path.join(out_dir, 'model_comparison.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Saved -> outputs/visualizations/model_comparison.png')

In [ ]:
# ============================================================
#  BEST MODEL DEEP DIVE
# ============================================================
from matplotlib.colors import LinearSegmentedColormap

# Fallback in case segmentation cell was skipped or failed
if 'seg_mean' not in dir():
    seg_mean = {'dice':0.0,'iou':0.0,'pixel_acc':0.0,'precision':0.0,'recall':0.0}
    seg_std  = {k:0.0 for k in seg_mean}
    print('WARNING: seg_mean not found — segmentation metrics will show 0.0')

best_name = best['name']
d = eval_data[best_name]
lbl, pred, prob = d['lbl'], d['pred'], d['prob']
fpr, tpr, roc_auc = d['fpr'], d['tpr'], d['roc']

acc       = accuracy_score(lbl, pred)
f1_macro  = f1_score(lbl, pred, average='macro')
f1_per    = f1_score(lbl, pred, average=None)
prec_macro= precision_score(lbl, pred, average='macro')
prec_per  = precision_score(lbl, pred, average=None)
rec_macro = recall_score(lbl, pred, average='macro')
rec_per   = recall_score(lbl, pred, average=None)
cm        = confusion_matrix(lbl, pred)

print(f'Best model: {best_name}')
print(classification_report(lbl, pred, target_names=CLASS_NAMES))

# ── Build full dashboard ──────────────────────────────────────
fig = plt.figure(figsize=(20, 14), facecolor='#0d1117')
fig.suptitle(f'Best Model Analysis  —  {best_name}',
             fontsize=16, color='#c9d1d9', fontweight='bold', y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Confusion matrix
ax1 = fig.add_subplot(gs[0, 0]); ax1.set_facecolor('#161b22')
cmap = LinearSegmentedColormap.from_list('cm', ['#161b22','#0d419d','#58a6ff'])
sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1,
            linewidths=0.5, linecolor='#21262d',
            annot_kws={'size':16,'weight':'bold','color':'#e6edf3'})
ax1.set_title('Confusion Matrix', color='#c9d1d9', pad=12, fontsize=13)
ax1.set_xlabel('Predicted', color='#8b949e'); ax1.set_ylabel('Actual', color='#8b949e')
corners = [('TP',GREEN),('FN',ACCENT),('FP',ACCENT),('TN',GREEN)]
for (lbl_c, col), (row, col_i) in zip(corners, [(0,0),(0,1),(1,0),(1,1)]):
    ax1.text(col_i+0.08, row+0.92, lbl_c, fontsize=9, color=col, fontweight='bold')

# 2. ROC curve
ax2 = fig.add_subplot(gs[0, 1]); ax2.set_facecolor('#161b22')
ax2.plot(fpr, tpr, color=BLUE, linewidth=2.5, label=f'AUC = {roc_auc:.4f}')
ax2.plot([0,1],[0,1], color='#444d56', linewidth=1, linestyle='--', label='Random')
ax2.fill_between(fpr, tpr, alpha=0.15, color=BLUE)
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve', color='#c9d1d9', pad=12, fontsize=13)
ax2.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9')
ax2.spines[:].set_edgecolor('#30363d')

# 3. Precision / Recall / F1 per class
ax3 = fig.add_subplot(gs[0, 2]); ax3.set_facecolor('#161b22')
x_bar = np.arange(len(CLASS_NAMES)); w = 0.25
for i, (vals, label, col) in enumerate([(prec_per,'Precision',BLUE),(rec_per,'Recall',YELLOW),(f1_per,'F1-Score',GREEN)]):
    bars = ax3.bar(x_bar + (i-1)*w, vals, w, label=label, color=col, alpha=0.9)
    for bar, v in zip(bars, vals):
        ax3.text(bar.get_x()+bar.get_width()/2, v+0.012, f'{v:.2f}',
                 ha='center', fontsize=9, color='#c9d1d9')
ax3.set_xticks(x_bar); ax3.set_xticklabels(CLASS_NAMES)
ax3.set_ylim(0, 1.15); ax3.set_title('Precision / Recall / F1', color='#c9d1d9', pad=12, fontsize=13)
ax3.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=10)
ax3.spines[:].set_edgecolor('#30363d')

# 4. Segmentation metrics
ax4 = fig.add_subplot(gs[1, 0]); ax4.set_facecolor('#161b22')
sk = ['dice','iou','pixel_acc','precision','recall']
slabels = ['Dice','IoU','Pix Acc','Precision','Recall']
svals = [seg_mean[k] for k in sk]
serrs = [seg_std[k]  for k in sk]
sbars = ax4.bar(slabels, svals, color=[GREEN,BLUE,PURPLE,YELLOW,ACCENT],
                alpha=0.9, yerr=serrs, capsize=5,
                error_kw={'ecolor':'#8b949e','elinewidth':1.5})
for bar, v in zip(sbars, svals):
    ax4.text(bar.get_x()+bar.get_width()/2, v+0.012, f'{v:.4f}',
             ha='center', fontsize=9, color='#c9d1d9', fontweight='bold')
ax4.set_ylim(0, 1.15); ax4.set_title('Segmentation Metrics (U-Net)', color='#c9d1d9', pad=12, fontsize=13)
ax4.spines[:].set_edgecolor('#30363d')

# 5. Confidence distribution
ax5 = fig.add_subplot(gs[1, 1]); ax5.set_facecolor('#161b22')
ax5.hist(prob[lbl==0], bins=20, alpha=0.75, color=ACCENT, label='True: Infected',     density=True)
ax5.hist(prob[lbl==1], bins=20, alpha=0.75, color=GREEN,  label='True: Non-Infected', density=True)
ax5.axvline(0.5, color=YELLOW, linewidth=1.5, linestyle='--', label='Decision boundary')
ax5.set_xlabel('P(Infected)'); ax5.set_ylabel('Density')
ax5.set_title('Score Distribution by Class', color='#c9d1d9', pad=12, fontsize=13)
ax5.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=9)
ax5.spines[:].set_edgecolor('#30363d')

# 6. Scorecard
ax6 = fig.add_subplot(gs[1, 2]); ax6.set_facecolor('#161b22'); ax6.axis('off')
ax6.set_title('Final Scorecard', color='#c9d1d9', pad=12, fontsize=13)

def grade(v):
    if v >= 0.90: return 'Excellent'
    if v >= 0.80: return 'Good'
    if v >= 0.65: return 'Fair'
    return 'Needs improvement'

rows = [
    ['Accuracy',           f'{acc*100:.2f}%',          grade(acc)],
    ['F1-Score (macro)',   f'{f1_macro:.4f}',           grade(f1_macro)],
    ['F1 — Infected',      f'{f1_per[0]:.4f}',         grade(f1_per[0])],
    ['F1 — Non-Infected',  f'{f1_per[1]:.4f}',         grade(f1_per[1])],
    ['Precision (macro)',  f'{prec_macro:.4f}',         grade(prec_macro)],
    ['Recall (macro)',     f'{rec_macro:.4f}',          grade(rec_macro)],
    ['AUC-ROC',            f'{roc_auc:.4f}',            grade(roc_auc)],
    ['Dice Score (U-Net)', f'{seg_mean["dice"]:.4f}',  grade(seg_mean['dice'])],
    ['IoU (U-Net)',        f'{seg_mean["iou"]:.4f}',   grade(seg_mean['iou'])],
]
tbl = ax6.table(cellText=rows, colLabels=['Metric','Value','Grade'],
                cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(9.5)
for (r,c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#21262d')
    if r == 0:
        cell.set_facecolor('#21262d')
        cell.set_text_props(color=BLUE, fontweight='bold')
    else:
        cell.set_facecolor('#1c2128' if r%2==0 else '#161b22')
        if c == 1:
            try:
                v = float(rows[r-1][1].replace('%',''))
                if '%' in rows[r-1][1]: v /= 100
                cell.set_text_props(color=GREEN if v>=0.80 else YELLOW if v>=0.65 else ACCENT)
            except: cell.set_text_props(color='#c9d1d9')
        else:
            cell.set_text_props(color='#c9d1d9')

out_dir = os.path.join(BASE, 'outputs', 'visualizations')
plt.savefig(os.path.join(out_dir, 'evaluation_dashboard.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Saved -> outputs/visualizations/evaluation_dashboard.png')

In [ ]:
# ============================================================
#  FINAL SUMMARY
# ============================================================
print('\n' + '='*62)
print('  CLASSIFICATION MODEL COMPARISON')
print('='*62)
print(f'  {"Model":<22} {"Acc%":>7} {"F1":>7} {"AUC":>7}  Winner?')
print('-'*62)
for r in results_table:
    star = '  <<< BEST' if r['name'] == best_name else ''
    print(f'  {r["name"]:<22} {r["acc"]*100:>7.2f} {r["f1"]:>7.4f} {r["auc"]:>7.4f}{star}')
print('='*62)
print(f'\n  SEGMENTATION (U-Net)')
print(f'  Dice Score : {seg_mean["dice"]:.4f}  ({grade(seg_mean["dice"])})')
print(f'  IoU        : {seg_mean["iou"]:.4f}  ({grade(seg_mean["iou"])})')
print(f'  Pixel Acc  : {seg_mean["pixel_acc"]:.4f}  ({grade(seg_mean["pixel_acc"])})')
print('='*62)
print(f'\n  BEST CLASSIFIER : {best_name}')
print(f'  Accuracy  : {best["acc"]*100:.2f}%  ({grade(best["acc"])})')
print(f'  F1-Score  : {best["f1"]:.4f}  ({grade(best["f1"])})')
print(f'  AUC-ROC   : {best["auc"]:.4f}  ({grade(best["auc"])})')
print('='*62)